# 100장 · 소유자·세션별 기록과 보존 한도

대화 이름이 같아도 소유자가 다르면 별도의 기록입니다. 사용자·세션을 함께 지정하고 보존할 턴 수와 초기화 범위를 정합니다.

두 사용자가 같은 세션 이름을 사용해도 각자 허용된 이력만 읽도록 저장 키를 설계합니다. 한 사용자의 기록을 삭제한 뒤 다른 사용자의 데이터가 남는지 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 복합 키로 대화 이력 조회

완성 상태표　가상 사용자 alice / 대화 main / 현재 근거 RULE-A. 가상 사용자 bob / 대화 main / 현재 근거 RULE-B. 예시에서는 신뢰한 수업용 주체와 세션 ID를 함께 키로 씁니다. 화면이 보내는 이름만 보고 소유자를 바꾸어서는 안 됩니다. 아래 메모리 사본으로 먼저 두 키를 읽어 보세요.

**복합 키와 세션 격리**　같은 main 세션 이름을 써도 alice는 RULE-A, bob은 RULE-B를 읽습니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter100/index.html#practice-space)


### 셀 01 · 복합 키로 세션 조회

두 사용자가 모두 main을 선택했을 때 bob의 현재 근거를 꺼냅니다.


In [1]:
sessions = {("alice", "main"): ["RULE-A"], ("bob", "main"): ["RULE-B"]}
trusted_owner = "bob"
print(sessions[(trusted_owner, "main")])

['RULE-B']


RULE-B만 나옵니다. 100장 제공 기록에서도 서로 다른 저장 파일에서 각각 RULE-A와 RULE-B를 읽었습니다. 이 셀은 같은 구분을 메모리 안에서만 보여 줍니다.

세션 이름만 같다고 같은 대화는 아닙니다. 이 예제는 소유자와 세션을 함께 지정해 RULE-B의 이력을 찾습니다. 삭제를 검증할 때도 지울 사용자의 데이터가 없어졌는지와 다른 사용자의 기록이 남았는지를 각각 확인합니다.


#### 새 답변이 와도 세션의 만료 시각이 그대로인 이유

```mermaid
flowchart TD
nborn["생성"]
  nclock["만료 시각 계산"]
  nturn["새 턴"]
  nexpire["만료"]
  nborn --> nclock
  nclock --> nturn
  nturn --> nexpire
```

보존 정책 예시에서 만료는 생성 후 300초입니다. 최근 다섯 턴을 관리하는 규칙과 만료 시각을 따로 추적하세요.

## 오래된 턴 정리와 선택 삭제

보존 정책 예시　생성 후 300초, 한 세션 안에서 최근 5턴(질문·응답 쌍)만 유지하고, 새 답이 생겨도 만료 시각은 연장하지 않습니다. 1000초에 만든 대화라면 1300초에 만료합니다. 삭제 대상은 현재 로컬 파일입니다. 로그와 백업 사본까지 지웠다고 기록하려면 그 사본도 별도로 확인해야 합니다.


### 과제 02 · 선택 연습 · 소유자별 이력 삭제

가상 alice가 main 삭제를 요청했습니다. 사본에서 그 키만 덜어내고 bob의 마지막 근거를 다시 읽어 보세요.

최근 5턴을 남기는 세션에서 여섯 번째 턴을 저장하면 첫 번째 턴이 빠지고 2~6번째 턴이 남습니다. 생성 후 300초가 되면 세션 자체가 만료됩니다. 아래 과제에서 실제로 관찰하는 것은 소유자를 골라 삭제한 결과입니다.


In [2]:
remaining = dict(sessions)
# alice/main을 지우고 bob/main을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>remaining = dict(sessions)
remaining.pop((&quot;alice&quot;, &quot;main&quot;), None)
print(remaining.get((&quot;alice&quot;, &quot;main&quot;)), remaining[(&quot;bob&quot;, &quot;main&quot;)])</code></pre>
<p>실행 결과</p><pre><code>None [&#x27;RULE-B&#x27;]</code></pre>
<p>None과 RULE-B가 보입니다. 한 사람의 삭제 요청을 대화 이름만으로 처리했다면 다른 사람의 기록까지 사라질 수 있습니다. 이번 과제의 완료 조건은 alice의 대화만 지운 뒤 bob의 대화가 남는지 확인하는 것입니다. 최대 5턴과 300초 만료는 설계 예시로 읽습니다.</p>
</details>


#### 한 사용자의 세션만 지울 때

<mark>세션 이름만 같아도 소유자가 다르면 다른 기록입니다.</mark> 두 사용자에게 같은 이름의 세션을 만들고 한쪽을 지운 뒤 양쪽을 조회하세요. 남아 있어야 하는 사용자의 RULE-B가 보존됐는지 마지막에 읽습니다.

선수 개념 · 60장 사용자별 장기 기억과 SQL 저장

## 세션의 마지막 다섯 턴과 최초 만료 시각을 따로 유지하기

두 사용자가 모두 main이라는 세션 이름을 씁니다. alice의 여섯 번째 질문이 들어올 때 가장 오래된 턴만 비우고, bob의 대화와 alice의 최초 만료 시각을 보존하는지 확인합니다.

### 셀 04 · 고정한 시각으로 경계 직전과 경계 시각 조회

시계 대신 초 단위 숫자를 넣습니다. alice의 대화는 1000초에 만들었고 수명은 300초입니다. 여섯 턴은 모두 만료 전에 도착했다고 가정합니다.

In [3]:
from collections import deque
store = {("alice", "main"): (1300, deque(maxlen=5)),
         ("bob", "main"): (1500, deque(["bob-r1"], maxlen=5))}
for number in range(1, 7):
    store[("alice", "main")][1].append(f"r{number}")

def read_turns(owner, session, now):
    expires, turns = store[(owner, session)]
    return list(turns) if now < expires else []

print(read_turns("alice", "main", 1299))
print(read_turns("alice", "main", 1300), read_turns("bob", "main", 1300))

['r2', 'r3', 'r4', 'r5', 'r6']
[] ['bob-r1']


1299초에는 r2부터 r6까지 남고 1300초가 되면 alice의 조회 결과만 빕니다. 이 셀은 만료 후 조회를 막습니다. 저장소에서 물리적으로 지우는 작업은 별도로 시험해야 합니다.

### 팀에서 한도를 고를 때

대화 한 턴을 사용자 질문과 답변 한 쌍으로 셀지 먼저 정하세요. 최근 세 턴이 필요한 팀이라면 deque의 한도를 3으로 바꾸고, 마지막 턴을 쓰기 전후에 다른 사용자의 기록과 만료 시각을 각각 다시 확인합니다.

<mark>새 턴 저장이 보존 기간을 연장할지는 팀이 선택한 정책입니다.</mark> 이 예시에서는 여섯 번째 턴이 들어와도 alice의 만료 시각은 1300초입니다. 최근 다섯 턴만 읽는 규칙은 만료 검사와 독립적으로 작동합니다.

101장 새 연동 후보에는 꼭 필요한 세션 값만 넘기고, 기록의 소유자나 보존 조건을 임의로 바꾸지 않습니다.

## 돌아보기

101장에 전달할 상태 정책은 소유자와 세션, 생성 때부터의 유효 시간, 삭제 범위입니다. 교재의 로컬 JSON 기록에서는 파일 읽기·쓰기를 확인했고 실제 로그인·DB·동시 쓰기·백업 삭제는 별도 확인이 필요합니다.
